# Ember2: Code Analysis and Documentation Pipeline

This notebook demonstrates the complete pipeline for analyzing Python code from Jupyter notebooks, extracting components, generating detailed descriptions, and creating comprehensive walkthroughs.

## Pipeline Steps:
1. **Parse Python Code** - Extract Python code from .ipynb files
2. **Initial Analysis** - Analyze code structure and create block organization
3. **Component Extraction** - Extract classes, functions, and resolve dependencies
4. **Deep Descriptions** - Generate detailed descriptions with iterative improvement
5. **Walkthrough Generation** - Create comprehensive code walkthrough


In [38]:
# Configuration - Specify the notebook to analyze
INPUT_NOTEBOOK = 'tiny_demo.ipynb'  # Change this to analyze a different notebook

# Create output directory
import os
OUTPUT_DIR = 'ember_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Output files that will be generated (delete these for fresh runs)
base_name = INPUT_NOTEBOOK.replace(".ipynb", "")
OUTPUT_FILES = {
    'python_code': os.path.join(OUTPUT_DIR, f'{base_name}_python.py'),
    'markdown': os.path.join(OUTPUT_DIR, f'{base_name}_markdown.md'),
    'initial_analysis': os.path.join(OUTPUT_DIR, f'{base_name}_initial_analysis.json'),
    'components': os.path.join(OUTPUT_DIR, f'{base_name}_components.json'),
    'deep_descriptions': os.path.join(OUTPUT_DIR, f'{base_name}_deep_descriptions.json'),
    'walkthrough': os.path.join(OUTPUT_DIR, f'{base_name}_walkthrough.md')
}

print(f"Analyzing: {INPUT_NOTEBOOK}")
print(f"\nOutput directory: {OUTPUT_DIR}/")
print("\nOutput files that will be generated:")
for key, value in OUTPUT_FILES.items():
    print(f"  {key}: {value}")
print(f"\nTo run a fresh analysis, delete the '{OUTPUT_DIR}' folder or its contents.")

Analyzing: tiny_demo.ipynb

Output directory: ember_output/

Output files that will be generated:
  python_code: ember_output/tiny_demo_python.py
  markdown: ember_output/tiny_demo_markdown.md
  initial_analysis: ember_output/tiny_demo_initial_analysis.json
  components: ember_output/tiny_demo_components.json
  deep_descriptions: ember_output/tiny_demo_deep_descriptions.json
  walkthrough: ember_output/tiny_demo_walkthrough.md

To run a fresh analysis, delete the 'ember_output' folder or its contents.


In [39]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()
os.environ["TAVILY_API_KEY"] 
os.environ["OPENAI_API_KEY"] 
os.environ["LANGCHAIN_TRACING_V2"] 
os.environ["LANGCHAIN_PROJECT"] 
os.environ["LANGCHAIN_API_KEY"] 

'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'

In [40]:
from openai import OpenAI
from langsmith.wrappers import wrap_openai
from langsmith import traceable

openai_client = wrap_openai(OpenAI())

## Step 1: Parse Python Code from Jupyter Notebook

The first step is to extract Python code and markdown documentation from the input notebook.
This function:
- Reads the .ipynb JSON structure
- Separates code cells from markdown cells
- Preserves the original order and structure
- Writes Python code to a .py file and markdown to a .md file

In [41]:
import json
import sys

def separate_ipynb(input_file, output_py=None, output_md=None):
    """
    Separate an ipynb file into .py and .md files while preserving order.
    
    Args:
        input_file: Path to the input .ipynb file
        output_py: Path for the output .py file (defaults to input_file.py)
        output_md: Path for the output .md file (defaults to input_file.md)
    """
    # Set default output filenames if not provided
    if output_py is None:
        output_py = input_file.replace('.ipynb', '.py')
    if output_md is None:
        output_md = input_file.replace('.ipynb', '.md')
    
    # Read the notebook
    with open(input_file, 'r', encoding='utf-8') as f:
        notebook = json.load(f)
    
    py_content = []
    md_content = []
    
    # Add header comments
    py_content.append(f"# Generated from {input_file}")
    py_content.append("# Python cells extracted in order\n")
    
    md_content.append(f"# Generated from {input_file}")
    md_content.append("## Markdown cells extracted in order\n")
    
    # Process cells in order
    for i, cell in enumerate(notebook.get('cells', [])):
        cell_type = cell.get('cell_type', '')
        source = cell.get('source', [])
        
        # Join source lines if it's a list
        if isinstance(source, list):
            source_text = ''.join(source)
        else:
            source_text = source
        
        if cell_type == 'code':
            # Add cell separator comment
            py_content.append(f"\n# ========== Cell {i+1} ==========")
            py_content.append(source_text.rstrip())
            
        elif cell_type == 'markdown':
            # Add cell separator
            md_content.append(f"\n---\n<!-- Cell {i+1} -->\n")
            md_content.append(source_text.rstrip())
    
    # Write the .py file
    with open(output_py, 'w', encoding='utf-8') as f:
        f.write('\n'.join(py_content))
    
    # Write the .md file
    with open(output_md, 'w', encoding='utf-8') as f:
        f.write('\n'.join(md_content))
        
    extracted_code = '\n'.join(py_content)
    extracted_markdown = '\n'.join(md_content)
    
    print(f"Successfully separated '{input_file}' into:")
    print(f"  - Python file: {output_py}")
    print(f"  - Markdown file: {output_md}")
    
    return extracted_code, extracted_markdown

# Example usage:
# separate_ipynb('example.ipynb')
# or with custom output names:
# separate_ipynb('example.ipynb', 'my_code.py', 'my_docs.md')

In [42]:
# Execute: Parse Python code from the input notebook
python_code, markdown_docs = separate_ipynb(
    INPUT_NOTEBOOK, 
    OUTPUT_FILES['python_code'], 
    OUTPUT_FILES['markdown']
)

print(f"Extracted {len(python_code.splitlines())} lines of Python code")
print(f"Extracted {len(markdown_docs.splitlines())} lines of markdown documentation")
print(f"\nPython code saved to: {OUTPUT_FILES['python_code']}")
print(f"Markdown saved to: {OUTPUT_FILES['markdown']}")

# Show a preview
print("\n--- Python Code Preview (first 20 lines) ---")
print('\n'.join(python_code.splitlines()[:20]))
print("..." if len(python_code.splitlines()) > 20 else "")

Successfully separated 'tiny_demo.ipynb' into:
  - Python file: ember_output/tiny_demo_python.py
  - Markdown file: ember_output/tiny_demo_markdown.md
Extracted 50 lines of Python code
Extracted 9 lines of markdown documentation

Python code saved to: ember_output/tiny_demo_python.py
Markdown saved to: ember_output/tiny_demo_markdown.md

--- Python Code Preview (first 20 lines) ---
# Generated from tiny_demo.ipynb
# Python cells extracted in order


# ========== Cell 2 ==========
# Import necessary libraries
import pandas as pd
from langchain import OpenAI
import numpy as np

# ========== Cell 3 ==========
# Helper function to clean text
def clean_text(text):
    """Remove extra whitespace and normalize text."""
    return ' '.join(text.split())

# Another helper
def tokenize(text):
    """Simple tokenization."""
    return text.lower().split()
...


## Step 2: Initial Analysis and Block Structure

Next, we analyze the code to understand its structure and create a hierarchical block organization.
This step:
- Identifies main components (classes, functions, global code)
- Creates a logical flow structure
- Tracks dependencies between components
- Provides initial explanations for each block

In [43]:
INITIAL_EXPLANATION_PROMPT = """You are an expert Python tutor analyzing the attached Python code to create a comprehensive explanation graph.

Your task is to decompose the code into logical blocks that form a dependency graph, explaining how each block functions and relates to others.

## Guidelines:
1. Each block should represent a cohesive unit of functionality
2. Blocks can contain: imports, class definitions, function definitions, global variables, or main execution code
3. Edge directions indicate dependencies: A -> B means "A depends on/uses B"
4. Bidirectional edges (A <-> B) indicate mutual dependencies
5. Every line of code must belong to exactly one block
6. Preserve the complete code in each block - do not split nested structures

## Target audience:
Someone with basic Python knowledge who needs to understand:
- What each section of code does
- How different parts interact
- The overall data and control flow

## Required JSON output structure:
{{
  "overview": "A 2-3 sentence summary of the code's overall purpose and functionality",
  "blocks": [
    {{
      "id": "1",
      "name": "Descriptive Block Name",
      "description": "Detailed explanation of what this block does, its role in the application, and key components within it. Include how it processes data or provides functionality.",
      "content": "# The actual Python code for this block\\n# Including all related lines",
      "dependencies": ["2", "3"],
      "dependents": ["4"]
    }}
  ],
  "edges": [
    {{"from": "1", "to": "2", "type": "uses"}},
    {{"from": "3", "to": "1", "type": "uses"}},
    {{"from": "4", "to": "5", "type": "bidirectional"}}
  ],
  "execution_order": ["1", "2", "3", "4", "5"]
}}

## Block identification hints:
- Group imports together if they serve similar purposes
- Keep class definitions with their methods
- Group related utility functions
- Separate initialization/configuration from main logic
- Consider data flow when deciding block boundaries

Python Code to analyze:

{code}
"""

In [44]:
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any

class CodeBlock(BaseModel):
    """Represents a block of code with its explanation and dependencies"""
    id: str = Field(description="Unique identifier for the block")
    type: str = Field(description="Type of block: setup, class, function, analysis, etc.")
    name: str = Field(description="Name/title of the block")
    code: str = Field(description="The actual code in this block")
    explanation: str = Field(description="Detailed explanation of what this code does")
    dependencies: List[str] = Field(default_factory=list, description="IDs of blocks this depends on")
    dependents: List[str] = Field(default_factory=list, description="IDs of blocks that depend on this")

class CodeExplanation(BaseModel):
    """Complete explanation of the code structure"""
    overview: str = Field(description="High-level overview of the entire code")
    main_components: List[str] = Field(description="List of main components/features")
    execution_flow: List[str] = Field(description="Step-by-step execution flow")
    blocks: List[CodeBlock] = Field(description="Detailed breakdown of code blocks")

In [45]:
# Execute: Initial Analysis and Block Structure
import json
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Initialize the model
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Create the analysis chain
analysis_chain = (
    ChatPromptTemplate.from_template(INITIAL_EXPLANATION_PROMPT)
    | model.with_structured_output(CodeExplanation)
)

# Check if analysis already exists
import os
if os.path.exists(OUTPUT_FILES['initial_analysis']):
    print(f"Loading existing analysis from {OUTPUT_FILES['initial_analysis']}")
    with open(OUTPUT_FILES['initial_analysis'], 'r') as f:
        explanation_data = json.load(f)
    initial_explanation = CodeExplanation(**explanation_data)
else:
    print("Generating initial analysis...")
    initial_explanation = analysis_chain.invoke({"code": python_code})
    # Save the analysis
    with open(OUTPUT_FILES['initial_analysis'], 'w') as f:
        json.dump(initial_explanation.model_dump(), f, indent=2)
    print(f"Analysis saved to {OUTPUT_FILES['initial_analysis']}")

# Display results
print("\n=== Code Overview ===")
print(initial_explanation.overview)
print("\n=== Main Components ===")
for i, comp in enumerate(initial_explanation.main_components, 1):
    print(f"{i}. {comp}")
print(f"\n=== Found {len(initial_explanation.blocks)} code blocks ===")
for block in initial_explanation.blocks[:3]:  # Show first 3 blocks
    print(f"\n[{block.id}] {block.name} ({block.type})")
    print(f"Dependencies: {block.dependencies}")
    print(f"First 100 chars: {block.code[:100]}...")

Generating initial analysis...


KeyboardInterrupt: 

## Step 3: Component Extraction and Dependency Resolution

This step extracts individual components (classes, functions) and resolves their dependencies.
Key features:
- Extracts each class and function as a separate component
- Tracks imports and dependencies
- Identifies which components use which other components
- Creates a dependency graph for understanding code relationships

In [ ]:
class Component(BaseModel):
    """Represents a code component (class or function)"""
    name: str = Field(description="Component name")
    type: str = Field(description="Type: class or function")
    code: str = Field(description="Complete code including docstring")
    imports: List[str] = Field(default_factory=list, description="Required imports")
    dependencies: List[str] = Field(default_factory=list, description="Other components used")
    docstring: Optional[str] = Field(None, description="Extracted docstring")
    methods: Optional[List[str]] = Field(None, description="For classes: list of methods")
    parameters: Optional[List[str]] = Field(None, description="For functions: list of parameters")

class ExtractedComponents(BaseModel):
    """Collection of extracted components"""
    components: List[Component] = Field(description="All extracted components")
    global_imports: List[str] = Field(description="All import statements")
    execution_order: List[str] = Field(description="Suggested order for understanding")

In [ ]:
# Execute: Component Extraction
COMPONENT_EXTRACTION_PROMPT = """Extract all classes and functions from the provided Python code.

For each component, identify:
1. Name and type (class or function)
2. Complete code including docstring
3. Required imports
4. Dependencies on other components
5. For classes: list all methods
6. For functions: list all parameters

Also provide:
- All global import statements
- Suggested execution order for understanding the code

Code to analyze:
{code}
"""

# Create extraction chain
extraction_chain = (
    ChatPromptTemplate.from_template(COMPONENT_EXTRACTION_PROMPT)
    | model.with_structured_output(ExtractedComponents)
)

# Check if components already extracted
if os.path.exists(OUTPUT_FILES['components']):
    print(f"Loading existing components from {OUTPUT_FILES['components']}")
    with open(OUTPUT_FILES['components'], 'r') as f:
        components_data = json.load(f)
    extracted = ExtractedComponents(**components_data)
else:
    print("Extracting components...")
    extracted = extraction_chain.invoke({"code": python_code})
    # Save components
    with open(OUTPUT_FILES['components'], 'w') as f:
        json.dump(extracted.model_dump(), f, indent=2)
    print(f"Components saved to {OUTPUT_FILES['components']}")

# Display results
print(f"\n=== Extracted {len(extracted.components)} components ===")
for comp in extracted.components:
    deps = f" -> depends on: {', '.join(comp.dependencies)}" if comp.dependencies else ""
    print(f"- {comp.name} ({comp.type}){deps}")
    if comp.type == "class" and comp.methods:
        print(f"  Methods: {', '.join(comp.methods[:5])}{'...' if len(comp.methods) > 5 else ''}")

print(f"\n=== Execution Order ===")
for i, name in enumerate(extracted.execution_order[:10], 1):
    print(f"{i}. {name}")

Extracting components...
Components saved to ember_output/tiny_demo_components.json

=== Extracted 4 components ===
- clean_text (function)
- tokenize (function)
- VectorStore (class)
  Methods: __init__, add, search
- build_rag_graph (function) -> depends on: VectorStore, clean_text, tokenize

=== Execution Order ===
1. clean_text
2. tokenize
3. VectorStore
4. build_rag_graph


## LangGraph State and Node Architecture

### What is LangGraph?
LangGraph is a framework for building stateful, multi-agent LLM applications. It provides:
- **State Management**: Shared state that flows through all nodes
- **Node-based Architecture**: Each processing step is a node
- **Conditional Routing**: Decide next steps based on state
- **Parallel Processing**: Run multiple nodes concurrently

### Our Pipeline Architecture
```
┌─────────────┐     ┌──────────────┐     ┌────────────────┐
│   Parse     │ --> │   Analyze    │ --> │    Extract     │
│   Code      │     │   Structure  │     │   Components   │
└─────────────┘     └──────────────┘     └────────────────┘
                                                   │
                                                   ▼
┌─────────────┐     ┌──────────────┐     ┌────────────────┐
│ Generate    │ <-- │   Improve    │ <-- │   Generate     │
│ Walkthrough │     │ Descriptions │     │ Descriptions   │
└─────────────┘     └──────────────┘     └────────────────┘
```

### State Definition
The state carries all data through the pipeline:

In [ ]:
from typing import TypedDict, Annotated
from operator import add

class PipelineState(TypedDict):
    """State that flows through the entire pipeline"""
    # Input
    input_file: str
    
    # Parsed content
    python_code: str
    markdown_docs: str
    
    # Initial analysis
    initial_explanation: CodeExplanation
    
    # Extracted components
    components: ExtractedComponents
    
    # Component descriptions (accumulates with each iteration)
    component_descriptions: Annotated[List[Dict], add]
    
    # Improvement tracking
    improvement_iteration: int
    improvement_history: List[Dict]
    
    # Final outputs
    walkthrough: str
    
# Each node is a function that takes state and returns updated state
def parse_node(state: PipelineState) -> PipelineState:
    """Parse the input notebook"""
    python_code, markdown = separate_ipynb(state['input_file'])
    return {
        **state,
        'python_code': python_code,
        'markdown_docs': markdown
    }

## Component Link Mapping and Naming

### How Component Linking Works

1. **Unique Identification**: Each component gets a unique name based on its type and position
   - Classes: `ClassName`
   - Functions: `function_name`
   - Methods: `ClassName.method_name`

2. **Dependency Resolution**: The system tracks:
   - **Direct Dependencies**: When one component directly calls another
   - **Import Dependencies**: What each component needs to import
   - **Inheritance**: Parent-child class relationships

3. **Link Generation**: In the final walkthrough, components are linked using:
   - Markdown anchors: `[ComponentName](#componentname)`
   - Consistent naming throughout the document
   - Bidirectional references (component to its dependencies and vice versa)

### Example Mapping
```python
# Component: DataProcessor (class)
# Dependencies: [pandas, numpy, BaseProcessor]
# Dependents: [analyze_data, create_report]

# In walkthrough, this becomes:
# - Links to BaseProcessor parent class
# - Referenced by analyze_data function
# - Methods linked as DataProcessor.process, DataProcessor.validate
```

## Step 4: Deep Description Generation with Iterative Improvement

This step generates detailed descriptions for each component and iteratively improves them.

### Initial Generation
- Creates comprehensive descriptions for each component
- Explains purpose, parameters, return values, and usage
- Identifies key algorithms and design patterns

### Iterative Improvement (3 iterations)
- Reviews all descriptions for consistency
- Adds cross-references between related components
- Enhances technical depth and clarity
- Ensures proper component linking

In [ ]:
class ComponentDescription(BaseModel):
    """Detailed description of a component"""
    name: str = Field(description="Component name")
    type: str = Field(description="Component type")
    purpose: str = Field(description="What this component does and why")
    detailed_explanation: str = Field(description="In-depth explanation")
    parameters: Optional[Dict[str, str]] = Field(None, description="Parameter descriptions")
    returns: Optional[str] = Field(None, description="Return value description")
    key_features: List[str] = Field(default_factory=list, description="Notable features")
    usage_example: Optional[str] = Field(None, description="How to use this component")
    related_components: List[str] = Field(default_factory=list, description="Related components")

class ImprovedDescriptions(BaseModel):
    """Collection of improved component descriptions"""
    descriptions: List[ComponentDescription]
    improvements_made: List[str] = Field(description="What was improved in this iteration")
    overall_quality_score: float = Field(description="Quality score 0-10")

In [ ]:
# Execute: Deep Description Generation
DEEP_DESCRIPTION_PROMPT = """Generate detailed descriptions for each component.

For each component, provide:
1. Clear purpose and responsibility
2. Detailed technical explanation
3. Parameter descriptions (if applicable)
4. Return value description (if applicable)
5. Key features and algorithms used
6. Usage example
7. Related components

Components to describe:
{components}

Code context:
{code}
"""

# Create description chain
description_chain = (
    ChatPromptTemplate.from_template(DEEP_DESCRIPTION_PROMPT)
    | model.with_structured_output(ImprovedDescriptions)
)

# Check if descriptions already exist
if os.path.exists(OUTPUT_FILES['deep_descriptions']):
    print(f"Loading existing descriptions from {OUTPUT_FILES['deep_descriptions']}")
    with open(OUTPUT_FILES['deep_descriptions'], 'r') as f:
        descriptions_data = json.load(f)
    final_descriptions = descriptions_data
else:
    print("Generating initial descriptions...")
    components_json = json.dumps([c.model_dump() for c in extracted.components], indent=2)
    
    # Initial generation
    descriptions = description_chain.invoke({
        "components": components_json,
        "code": python_code
    })
    
    print(f"Initial quality score: {descriptions.overall_quality_score}/10")
    
    # Iterative improvement
    improvement_history = [descriptions.model_dump()]
    
    IMPROVEMENT_PROMPT = """Review and improve these component descriptions.

Current descriptions:
{current_descriptions}

Improve by:
1. Adding more technical depth
2. Ensuring consistency across descriptions
3. Adding cross-references between related components
4. Clarifying any ambiguous explanations
5. Enhancing usage examples

Original code for reference:
{code}
"""
    
    improvement_chain = (
        ChatPromptTemplate.from_template(IMPROVEMENT_PROMPT)
        | model.with_structured_output(ImprovedDescriptions)
    )
    
    # 3 improvement iterations
    for i in range(3):
        print(f"\nImprovement iteration {i+1}...")
        current_json = json.dumps(descriptions.model_dump(), indent=2)
        descriptions = improvement_chain.invoke({
            "current_descriptions": current_json,
            "code": python_code
        })
        print(f"Quality score: {descriptions.overall_quality_score}/10")
        print(f"Improvements: {', '.join(descriptions.improvements_made[:3])}...")
        improvement_history.append(descriptions.model_dump())
    
    # Save final descriptions
    final_descriptions = {
        "final": descriptions.model_dump(),
        "improvement_history": improvement_history
    }
    with open(OUTPUT_FILES['deep_descriptions'], 'w') as f:
        json.dump(final_descriptions, f, indent=2)
    print(f"\nDescriptions saved to {OUTPUT_FILES['deep_descriptions']}")

# Display sample descriptions
print("\n=== Sample Component Descriptions ===")
final_descs = final_descriptions['final']['descriptions'] if 'final' in final_descriptions else final_descriptions['descriptions']
for desc in final_descs[:2]:
    print(f"\n{desc['name']} ({desc['type']})")
    print(f"Purpose: {desc['purpose']}")
    print(f"Key features: {', '.join(desc['key_features'][:3])}")
    if desc['related_components']:
        print(f"Related: {', '.join(desc['related_components'])}")

Generating initial descriptions...
Initial quality score: 9.0/10

Improvement iteration 1...
Quality score: 9.5/10
Improvements: Enhanced technical depth in explanations and descriptions., Ensured consistency in structure and terminology across all component descriptions., Added cross-references between related components for better context....

Improvement iteration 2...
Quality score: 9.5/10
Improvements: Enhanced technical depth in explanations and descriptions., Ensured consistency in structure and terminology across all component descriptions., Added cross-references between related components for better context....

Improvement iteration 3...
Quality score: 9.5/10
Improvements: Enhanced technical depth in explanations and descriptions., Ensured consistency in structure and terminology across all component descriptions., Added cross-references between related components for better context....

Descriptions saved to ember_output/tiny_demo_deep_descriptions.json

=== Sample Componen

## Step 5: Comprehensive Walkthrough Generation

The final step creates a complete, well-structured walkthrough document that:
- Provides a high-level overview
- Explains the code flow and architecture
- Details each component with proper linking
- Includes usage examples and best practices
- Creates a table of contents with navigation

In [ ]:
# Execute: Walkthrough Generation
WALKTHROUGH_PROMPT = """Create a comprehensive walkthrough document for this codebase.

Structure the walkthrough as follows:
1. Executive Summary
2. Table of Contents
3. Overview and Architecture
4. Component Details (with proper markdown linking)
5. Usage Guide
6. Code Flow and Examples

Use markdown formatting with:
- Clear headings (##, ###)
- Code blocks with syntax highlighting
- Links between components using [ComponentName](#componentname)
- Tables where appropriate

Components and descriptions:
{descriptions}

Initial analysis:
{initial_analysis}

Markdown documentation:
{markdown_docs}
"""

# Create walkthrough chain
walkthrough_chain = (
    ChatPromptTemplate.from_template(WALKTHROUGH_PROMPT)
    | model
)

# Check if walkthrough already exists
if os.path.exists(OUTPUT_FILES['walkthrough']):
    print(f"Loading existing walkthrough from {OUTPUT_FILES['walkthrough']}")
    with open(OUTPUT_FILES['walkthrough'], 'r') as f:
        walkthrough_content = f.read()
else:
    print("Generating comprehensive walkthrough...")
    
    # Prepare data
    desc_json = json.dumps(final_descriptions['final'] if 'final' in final_descriptions else final_descriptions, indent=2)
    analysis_json = json.dumps(initial_explanation.model_dump(), indent=2)
    
    # Generate walkthrough
    response = walkthrough_chain.invoke({
        "descriptions": desc_json,
        "initial_analysis": analysis_json,
        "markdown_docs": markdown_docs
    })
    
    walkthrough_content = response.content
    
    # Save walkthrough
    with open(OUTPUT_FILES['walkthrough'], 'w') as f:
        f.write(walkthrough_content)
    print(f"Walkthrough saved to {OUTPUT_FILES['walkthrough']}")

# Display preview
print("\n=== Walkthrough Preview (first 500 chars) ===")
print(walkthrough_content[:500] + "...")
print(f"\nTotal length: {len(walkthrough_content)} characters")
print(f"\nView the complete walkthrough in: {OUTPUT_FILES['walkthrough']}")

Generating comprehensive walkthrough...
Walkthrough saved to ember_output/tiny_demo_walkthrough.md

=== Walkthrough Preview (first 500 chars) ===
# Comprehensive Walkthrough Document for the Codebase

## Executive Summary
This document provides a detailed walkthrough of a codebase designed for processing text documents into a vector store, facilitating retrieval-augmented generation (RAG) tasks. The codebase includes functions for text cleaning and tokenization, a class for managing vector storage, and a main function to build the RAG graph from a list of documents. This walkthrough aims to guide developers and users through the architect...

Total length: 7414 characters

View the complete walkthrough in: ember_output/tiny_demo_walkthrough.md


## Summary and File Management

### Generated Files
The pipeline generates the following files based on your input notebook:

1. **Python Code**: `{input}_python.py` - Extracted Python code
2. **Markdown**: `{input}_markdown.md` - Extracted documentation
3. **Initial Analysis**: `{input}_initial_analysis.json` - Code structure analysis
4. **Components**: `{input}_components.json` - Extracted components and dependencies
5. **Deep Descriptions**: `{input}_deep_descriptions.json` - Detailed component descriptions with improvement history
6. **Walkthrough**: `{input}_walkthrough.md` - Complete documentation

### Running Fresh Analysis
To run a fresh analysis on the same notebook:
1. Delete all the output files listed above
2. Re-run all cells in order

### Changing Input Notebook
1. Update `INPUT_NOTEBOOK` in the configuration cell
2. Re-run all cells (previous outputs will be preserved)

### Performance Notes
- The pipeline caches results at each step
- If a step's output file exists, it will be loaded instead of regenerated
- This allows you to iterate on later steps without re-running earlier ones